# 🎤 AI Vocals Studio — Voice Model Training

**Before running:** `Runtime` → `Change runtime type` → **T4 GPU** → Save

Then click **Runtime → Run all** and follow the Google Drive auth prompt.

That's it — everything else is automatic.

In [ ]:
# ── Step 1: Check GPU ────────────────────────────────────────────
import torch
if torch.cuda.is_available():
    print(f'✅ GPU ready: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    raise RuntimeError('❌ No GPU found! Go to Runtime → Change runtime type → T4 GPU, then Run All again.')

In [ ]:
# ── Step 2: Install dependencies ─────────────────────────────────
!pip install -q so-vits-svc-fork==4.2.30 torchcodec
print('✅ Dependencies installed')

In [ ]:
# ── Step 3: Mount Google Drive and find your zip ──────────────────
from google.colab import drive
drive.mount('/content/drive')

import glob, os, zipfile

# Auto-find any *_colab_training.zip or *training*.zip in Drive
search_patterns = [
    '/content/drive/MyDrive/**/*_colab_training.zip',
    '/content/drive/MyDrive/**/*training*.zip',
    '/content/drive/MyDrive/**/*.zip',
]
found = []
for pat in search_patterns:
    found = glob.glob(pat, recursive=True)
    if found:
        break

if not found:
    raise FileNotFoundError(
        'No zip found in Google Drive.\n'
        'Upload your *_colab_training.zip to Google Drive first, then re-run this cell.'
    )

# Use most recently modified zip
zip_path = max(found, key=os.path.getmtime)
zip_size = os.path.getsize(zip_path) / 1e9
print(f'✅ Found zip: {os.path.basename(zip_path)} ({zip_size:.2f} GB)')

print('📦 Extracting...')
with zipfile.ZipFile(zip_path) as z:
    z.extractall('/content/')
print('✅ Extracted to /content/')

In [ ]:
# ── Step 4: Verify dataset ────────────────────────────────────────
import glob as g
features = g.glob('/content/dataset/44k/**/*.data.pt', recursive=True)
wavs     = g.glob('/content/dataset/44k/**/*.wav', recursive=True)
configs  = g.glob('/content/configs/44k/config.json')
ckpts    = g.glob('/content/logs/44k/G_0.pth')

print(f'Feature files (.data.pt): {len(features)}')
print(f'WAV files:                {len(wavs)}')
print(f'Config found:             {bool(configs)}')
print(f'Base checkpoint (G_0):    {bool(ckpts)}')

if not features:
    raise RuntimeError('No .data.pt feature files found. The zip may be incomplete.')
if not configs:
    raise RuntimeError('No config.json found. The zip may be incomplete.')

print('\n✅ Dataset looks good — ready to train!')

In [ ]:
# ── Step 5: Train ─────────────────────────────────────────────────
# This cell runs for several hours. Progress prints every 10 steps.
# Colab will keep running even if you close the browser tab.
import os
os.chdir('/content')

print('🚀 Starting training... (grab a coffee, this takes 6-12 hours)')
!svc train -c configs/44k/config.json -m logs/44k

In [ ]:
# ── Step 6: Download trained model ────────────────────────────────
import glob, os
from google.colab import files

# Find the latest checkpoint (highest step number)
checkpoints = sorted(
    glob.glob('/content/logs/44k/G_*.pth'),
    key=lambda p: int(p.split('G_')[1].replace('.pth','')) if p.split('G_')[1].replace('.pth','').isdigit() else 0
)
print('All checkpoints:', [os.path.basename(c) for c in checkpoints])

if not checkpoints:
    print('⚠️  No trained checkpoints found. Training may not have completed.')
else:
    best = checkpoints[-1]
    print(f'\n✅ Downloading: {os.path.basename(best)}')
    files.download(best)
    files.download('/content/logs/44k/config.json')
    print('\n🎉 Done! Install with: bash cloud_train.sh --install')

In [ ]:
# ── Optional: Save checkpoint to Google Drive ─────────────────────
# Run this cell if the download above didn't work
import glob, os, shutil
checkpoints = sorted(glob.glob('/content/logs/44k/G_*.pth'))
if checkpoints:
    best = checkpoints[-1]
    save_dir = '/content/drive/MyDrive/ai_vocals_trained_model'
    os.makedirs(save_dir, exist_ok=True)
    shutil.copy(best, save_dir)
    shutil.copy('/content/logs/44k/config.json', save_dir)
    print(f'✅ Saved to Google Drive: {save_dir}')
    print(f'   Files: {os.listdir(save_dir)}')